## import libraries

In [1]:
platformID = 'YT-'

In [2]:
from IPython.display import display

import os
import zipfile

from tqdm import tqdm 
from datetime import datetime

import pandas as pd
pd.set_option('display.max_colwidth', None)

import numpy as np

import re

#import yxdb

import missingno as msno
import matplotlib.pyplot as plt
import seaborn as sns 

import psycopg2

## import helper 

In [3]:
import sys
from pathlib import Path

try:
    # Works in Python scripts
    helper_path = Path(__file__).resolve().parent.parent / "helper"
except NameError:
    # Works in Jupyter notebooks
    helper_path = Path().resolve().parent / "helper"

sys.path.insert(0, str(helper_path))

# Now import your modules 
from config import gam_info

from functions import lookup_loader, execute_sql_query
import test_functions

In [4]:
lookup = lookup_loader(gam_info, platformID, '1c',
                       with_country=True, country_col=['YT-_FBE_codes'])
week_tester = lookup['week_tester']
socialmedia_accounts = lookup['socialmedia_accounts']
country_codes = lookup['country_codes']


✅ Test YT-_1c_00 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test YT-_1c_01 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test YT-_1c_02 passed: No missing values in lookup.
...updating logbook...

✅ Test YT-_1c_03 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test YT-_1c_04 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test YT-_1c_05 passed: No missing values in lookup.
...updating logbook...

✅ Test YT-_1c_06 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test YT-_1c_07 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test YT-_1c_08 passed: No missing values in lookup.
...updating logbook...



'# country\ncountry_cols = [\'PlaceID\', \'YT-_FBE_codes\']\ncountry_codes = pd.read_excel(f"../../{gam_info[\'lookup_file\']}", sheet_name=\'CountryID\', \n                              keep_default_na=False)\n\n# week \nweek_tester = pd.read_excel(f"../../{gam_info[\'lookup_file\']}", sheet_name=\'GAM Period\')\nweek_tester[\'w/c\'] = pd.to_datetime(week_tester[\'w/c\'])\n\n#\xa0social media accounts\ndtype_dict = {\'Channel ID\': \'str\',\n              \'Linked FB Account\': \'str\'}\nsocialmedia_accounts = pd.read_excel(f"../../{gam_info[\'lookup_file\']}", dtype=dtype_dict,\n                                     sheet_name=\'Social Media Accounts new\')\n\nsocialmedia_accounts = socialmedia_accounts[(socialmedia_accounts[\'PlatformID\'] == platformID)\n                                            & \n                                            (socialmedia_accounts[\'Status\'] == \'active\')]\nsocialmedia_accounts = socialmedia_accounts.rename(columns={\'Excluding UK\': \'Channel G

# automated 

In [5]:
channel_ids = socialmedia_accounts['Channel ID'].unique().tolist()

sql_query = f"""
    SELECT
        week_commencing,
        channel_id,
        channel_name,
        country_code,
        views_country
    FROM
        central_insights.adverity_social_youtube_by_channel_geo
    WHERE
        week_commencing BETWEEN '{gam_info['w/c_start']}' AND '{gam_info['w/c_end']}'
    ;
    """
file = f"../data/raw/{platformID}/{gam_info['file_timeinfo']}_{platformID}_country_redshift_extract.csv"

try: 
    df = execute_sql_query(sql_query)
    df['channel_id'] = platformID+df['channel_id'].astype(str)
    df.to_csv(file, index=False, na_rep='')
except:
    print("no redshift connection using last time queried")

yt_views_raw = pd.read_csv(file, keep_default_na=False)
yt_views_raw['week_commencing'] = pd.to_datetime(yt_views_raw['week_commencing'])
yt_views_raw['country_code'] = yt_views_raw['country_code'].replace('', 'ZZ')
yt_views_raw = yt_views_raw.rename(columns = {'week_commencing': 'w/c',
                                              'channel_id': 'Channel ID',
                                              'channel_name': 'Channel Name',
                                              'country_code': 'YT-_FBE_codes'})

In [6]:
yt_views_raw = yt_views_raw[yt_views_raw['Channel ID'].isin(channel_ids)]

### RUN TESTS


# missing page_ids
test_functions.test_filter_elements_returned(yt_views_raw, 
                                             channel_ids, 
                                             'Channel ID', 
                                             f"{platformID}_1c_09",
                                             "Check that all page IDs are found in SQL")


# missing weeks per page_id
test_functions.test_weeks_presence_per_account(key='w/c',
                                               channel_id_col=['Channel ID'],
                                               main_data=yt_views_raw,
                                               week_lookup=week_tester[['w/c']],
                                               channel_lookup=socialmedia_accounts[['Channel ID', 'Start', 'End']],
                                               test_number=f"{platformID}_1c_10",
                                               test_step="Check all weeks present for each account")

# missing values per week / page id 
test_functions.test_non_null_and_positive(yt_views_raw, 
                           numeric_columns=['views_country'], 
                           test_number=f"{platformID}_1c_11",
                           test_step='Check no missing values in page fans column from redshift returned')

# test for duplicate entries 
test_functions.test_duplicates(yt_views_raw, ['Channel ID', 'w/c', 'YT-_FBE_codes'], 
                               test_number=f"{platformID}_1c_12",
                               test_step='Check no duplicates from redshift returned')

...testing Channel ID...
✅ Test YT-_1c_09 passed: everything found!
...updating logbook...

❌ Missing weeks from Start onward:
          Start End                   Channel ID        w/c
76   2020-04-01 NaT  YT-UC0JypFmHP-9wh5A3JjJLpcA 2025-12-15
1001 2020-04-01 NaT  YT-UCHaHD477h-FeBbVh9Sh7syA 2025-12-15
728  2020-04-01 NaT  YT-UCd9maKo3B6jX8pCPzLa2hvA 2025-12-15
1040 2020-04-01 NaT  YT-UCHbI0mPbb_q9tokc5RkOONA 2025-12-15
1157 2020-04-01 NaT  YT-UCIDOGTbwTBHZ5YoR9Xjcp-w 2025-12-15
...         ...  ..                          ...        ...
1264 2020-04-01 NaT  YT-UCN0VXYoN2ZdC7s8OFC_zHCg 2025-12-22
1158 2020-04-01 NaT  YT-UCIDOGTbwTBHZ5YoR9Xjcp-w 2025-12-22
1041 2020-04-01 NaT  YT-UCHbI0mPbb_q9tokc5RkOONA 2025-12-22
651  2020-04-01 NaT  YT-UCcgKRB26m6M3okTSP7C_UwA 2025-12-22
2033 2020-04-01 NaT  YT-UCznzKevbLg9SjtfAbjXiNFw 2025-12-22

[108 rows x 4 columns]

Summary of missing weeks per channel_id_col:
                     Channel ID  missing_week_count
0   YT-UC0JypFmHP-9wh5A3JjJLpcA

In [7]:
# add PlaceID
cols = ['PlaceID', 'YT-_FBE_codes']
yt_views = yt_views_raw.merge(country_codes[cols],
                              on=['YT-_FBE_codes'],
                              how='left').drop(columns=['YT-_FBE_codes'])
test_functions.test_inner_join(yt_views_raw, 
                               country_codes[cols], 
                               ['YT-_FBE_codes'], 
                               f"{platformID}_1c_13",
                               test_step='adding country codes GAM', 
                               focus='left')



✅ Inner join test YT-_1c_13 successful: No issues found.
...updating logbook...



In [9]:
# Group by the specified columns and sum the yt_metric_value
yt_views_global = yt_views.groupby([ 'Channel ID','w/c']).agg({'views_country': 'sum'}).reset_index()
yt_views_global = yt_views_global.rename(columns={'views_country': 'total_views_country'})
#display(grouped_df_allCountries.sample())

yt_country = yt_views.merge(yt_views_global,
                                    on=['Channel ID', 'w/c'],
                                    how='inner')
test_functions.test_inner_join(yt_views, 
                               yt_views_global, 
                               ['Channel ID', 'w/c'],
                               f"{platformID}_1c_14",
                               test_step='combining country and global views', 
                               focus='both')


yt_country['country_%'] = (yt_country['views_country'] / yt_country['total_views_country'])

test_functions.test_percentage(yt_country,  
                               ['Channel ID', 'w/c'], 
                               f"{platformID}_1c_15",
                               test_step = 'calculating % country')
yt_country

✅ Inner join test YT-_1c_14 successful: No issues found.
...updating logbook...

...updating logbook...



,w/c,Channel ID,Channel Name,views_country,PlaceID,total_views_country,country_%
0,2025-11-17,YT-UC0JypFmHP-9wh5A3JjJLpcA,BBC News Azərbaycanca,790,UAE,1435013,0.000551
1,2025-11-17,YT-UC46q-QSvoJz-1iSxPeuOqWA,BBC News Indonesia,429,UK,797359,0.000538
2,2025-11-17,YT-UC8zQiuT0m1TELequJ5sp5zw,BBC News - Русская служба,5765,CAN,1433666,0.004021
3,2025-11-17,YT-UCBte7YLdJx-O_YljuvN6whg,BBC Afrique,92,FPO,139793,0.000658
4,2025-11-17,YT-UCF4QKhPMP1JybbkIJzIayww,BBC News India,461355,IND,674005,0.684498
...,...,...,...,...,...,...,...
153369,2025-12-08,YT-UCBwejc-awS-mANgrkHQ5-6A,BBC London,99,UAE,24726,0.004004
153370,2025-12-08,YT-UCBwejc-awS-mANgrkHQ5-6A,BBC London,40,ARG,24726,0.001618
153371,2025-12-08,YT-UCBwejc-awS-mANgrkHQ5-6A,BBC London,48,OST,24726,0.001941
153372,2025-12-08,YT-UCBwejc-awS-mANgrkHQ5-6A,BBC London,516,AUS,24726,0.020869


# manual

# store

In [10]:

file_path = f"../data/processed/{platformID}"
os.makedirs(file_path, exist_ok=True)

cols = ['Channel ID', 'Channel Name', 'w/c', 'PlaceID', 'country_%']
yt_country[cols].to_csv(f"{file_path}/{gam_info['file_timeinfo']}_REDSHIFT_COUNTRY.csv", 
                         index=None, na_rep='')